# Public Portfolio Note

**Purpose:** Document the final data integration workflow used to combine matched Google-derived records, CMS hospital quality measures, AHRQ county SDOH indicators, and CMS POS teaching variables.

**Inputs:** Excluded intermediate match outputs plus separately obtained CMS, AHRQ, and CMS POS source files in the expected local filenames.

**Outputs:** Excluded staged datasets used by downstream preprocessing and modeling notebooks.

**Public Data Limitation:** The merged row-level analytic dataset is excluded because it contains Google API-derived fields and project-specific intermediate records.


# Section 4: Final Dataset Merge & Data Staging

This section documents the integration of the matched hospital crosswalk with Google-derived, CMS, AHRQ SDOH, and CMS POS fields. It also prepares lightweight staging tables for downstream feature engineering.

**Key Steps:**
1. Combine stage-1 matches with recovered unmatched records.
2. Join CMS hospital records to matched Google-derived records.
3. Build the merged hospital dataset used by preprocessing.
4. Stage California county-level SDOH indicators.
5. Stage CMS POS teaching variables for academic medical center flags.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import os
# Public copy: row-level preview/debug output removed.


In [ ]:

# Public path constants. These row-level source and intermediate files are excluded from the public repository.
RECOVERED_UNMATCHED_GOOGLE_RECORDS_PATH = "data/excluded/recovered_unmatched_google_places_records.csv"
MATCHED_HOSPITAL_CROSSWALK_PATH = "data/excluded/matched_hospital_crosswalk.csv"
CMS_HOSPITAL_QUALITY_RECORDS_PATH = "data/excluded/cms_hospital_quality_records.csv"
GOOGLE_PLACES_CLEANED_RECORDS_PATH = "data/excluded/google_places_hospitals_cleaned.csv"
MERGED_HOSPITAL_DATASET_PATH = "data/excluded/merged_hospital_dataset.csv"
AHRQ_SDOH_SOURCE_PATH = "data/excluded/ahrq_sdoh_2022.xlsx"
CA_COUNTY_SDOH_SUBSET_PATH = "data/excluded/ca_county_sdoh_subset.csv"
CMS_PROVIDER_OF_SERVICES_SOURCE_PATH = "data/excluded/cms_provider_of_services_q4_2025.csv"
CMS_POS_TEACHING_INDICATORS_PATH = "data/excluded/cms_pos_teaching_indicators.csv"


## 4.1 Final Hospital Dataset

In [ ]:
# 1. load dataset
df_rescued = pd.read_csv(RECOVERED_UNMATCHED_GOOGLE_RECORDS_PATH)
df_matched = pd.read_csv(MATCHED_HOSPITAL_CROSSWALK_PATH)
df_cms = pd.read_csv(CMS_HOSPITAL_QUALITY_RECORDS_PATH)
df_google = pd.read_csv(GOOGLE_PLACES_CLEANED_RECORDS_PATH)

# 2. find cms_id
df_cms_ca_only = df_cms.copy()
df_cms_ca_only['zip5'] = df_cms_ca_only['cms_zip'].astype(str).str[:5]

name_zip_to_id = dict(zip(
    df_cms_ca_only['cms_name'] + "_" + df_cms_ca_only['zip5'], 
    df_cms_ca_only['cms_id']
))

df_rescued['zip5'] = df_rescued['cms_zip'].astype(str).str[:5]
df_rescued['cms_id'] = (df_rescued['cms_name'] + "_" + df_rescued['zip5']).map(name_zip_to_id)

# 3. match all samples

df_rescued_google = df_rescued.rename(columns={
    'place_id': 'google_id',
    'hospital_name': 'google_name',
    'rating': 'google_star',
    'rating_count': 'google_rating_count',
    'address': 'google_add',
    'county': 'google_county',
    'lat': 'latitude',   
    'lon': 'longitude'
})

df_google_expanded = pd.concat([df_google, df_rescued_google], ignore_index=True).drop_duplicates(subset=['google_id'], keep='last')
print(f"Google total length: {len(df_google_expanded)} ")

# stage 2 new matched data
df_rescued_bridge = df_rescued[['cms_id', 'place_id']].rename(columns={'place_id': 'google_id'})
df_rescued_bridge['match_score'] = 100.0  

# clean stage 1 matched data
df_matched_clean = df_matched.dropna(subset=['google_id'])

# final matched data
df_bridge_full = pd.concat([
    df_matched_clean[['cms_id', 'google_id', 'match_score']], 
    df_rescued_bridge
], ignore_index=True).dropna(subset=['cms_id']).drop_duplicates(subset=['cms_id'], keep='last')

print(f"Matched hospital: {len(df_bridge_full)} ")

# 4. final data set

df_final = (
    df_cms
    .merge(df_bridge_full, on='cms_id', how='inner')
    .merge(df_google_expanded[['google_id', 'google_name','google_star', 'google_rating_count', 'latitude', 'longitude']], 
           on='google_id', how='left')
)

print("-" * 50)
print(f"Final Dataset: {df_final.shape}")
print("-" * 50)

final_columns_order = [
    'cms_id', 'google_id', 'cms_name', 'google_name','cms_county', 'cms_zip', 
    'latitude', 'longitude', 
    'cms_star', 'clinical_score', 'patient_exp_score', 'total_score', 
    'google_star', 'google_rating_count'
]

df_final = df_final[final_columns_order]
df_final['cms_id'] = df_final['cms_id'].astype(str).str.split('.').str[0].str.zfill(6)
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this section produced:
#
# - merged_hospital_dataset.csv
#   Hospital-level merged dataset linking CMS quality records with matched Google Places-derived records.
#
# This file is excluded from the public repository because it contains
# row-level Google Places API-derived records and intermediate merge artifacts.
#
# Private workflow only:
# df_final.to_csv(MERGED_HOSPITAL_DATASET_PATH, index=False, encoding='utf-8-sig')

print("Public copy: merged hospital dataset CSV export disabled. Expected private output is documented above.")

# Public copy: df.info() output removed.
# Public copy: row-level preview/debug output removed.


## 4.2 SDOH Dataset

In [ ]:
# 1.choose interested SDOH indicators
sdoh_cols = ['COUNTY', 'STATE', 'ACS_GINI_INDEX', 'CDCSVI_RPL_THEMES_ALL']

# 2. load SDOH dataset 
df_sdoh = pd.read_excel(AHRQ_SDOH_SOURCE_PATH, sheet_name='Data', usecols=sdoh_cols)

# filter CA data
df_county_sdoh = df_sdoh[df_sdoh['STATE'] == 'California'].copy()

# 3. rename for preparation of merging
df_county_sdoh['sdoh_county'] = df_county_sdoh['COUNTY'].str.replace(' County', '', regex=False).str.upper() # turn county into upper form
df_county_sdoh.rename(columns={"ACS_GINI_INDEX": "Gini_idx",
                               "CDCSVI_RPL_THEMES_ALL": "SVI_idx"},inplace=True)

# 4. output
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this section produced:
#
# - ca_county_sdoh_subset.csv
#   California county-level SDOH staging table used by the preprocessing notebook.
#
# This file is excluded from the public repository as an intermediate staging artifact.
#
# Private workflow only:
# df_county_sdoh.to_csv(CA_COUNTY_SDOH_SUBSET_PATH, index=False)

print("Public copy: SDOH staging CSV export disabled. Expected private output is documented above.")

# Public copy: row-level preview/debug output removed.


## 4.3 CMS POS Dataset

In [ ]:
# 1. load CMS POS data
teaching_cols = [
    'PRVDR_NUM',          # CMS Facility ID
    'FAC_NAME',           # CMS name
    'STATE_CD',           # state
    'MDCL_SCHL_AFLTN_CD', # affiliation code
    'RSDNT_PHYSN_CNT'     # resident num
]

df_pos = pd.read_csv(CMS_PROVIDER_OF_SERVICES_SOURCE_PATH, usecols=teaching_cols, dtype={'PRVDR_NUM': str})

# 2. filter CA data
df_ca_teaching = df_pos[df_pos['STATE_CD'] == 'CA'].copy()

# 3. rename for preparation of merging
df_ca_teaching.rename(columns={'PRVDR_NUM': 'cms_id',
                               'FAC_NAME': 'cms_name'}, inplace=True)

# 4. output
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this section produced:
#
# - cms_pos_teaching_indicators.csv
#   CMS POS teaching-indicator staging table used to construct academic medical center flags.
#
# This file is excluded from the public repository as an intermediate staging artifact.
#
# Private workflow only:
# df_ca_teaching.to_csv(CMS_POS_TEACHING_INDICATORS_PATH, index=False)

print("Public copy: CMS POS teaching indicator CSV export disabled. Expected private output is documented above.")

# Public copy: row-level preview/debug output removed.
